# Исследование рынка общественного питания Москвы

## 0.1 Цели и задачи проекта

**Цель проекта** — провести исследовательский анализ рынка общественного питания Москвы, чтобы помочь инвесторам фонда Shut Up and Take My Money выбрать подходящее место для открытия нового заведения (кафе, ресторана или бара).

**Задачи проекта:**
- Загрузить данные и познакомиться с их содержимым.
- Провести предобработку данных.
- Провести исследовательский анализ данных:
  1. изучить распределение заведений по категориям и районам;
  2. исследовать соотношение сетевых и несетевых заведений;
  3. изучить количество посадочных мест и выявить выбросы;
  4. проанализировать рейтинги заведений и их взаимосвязи с другими признаками;
  5. найти топ-15 популярных сетей Москвы;
  6. изучить вариацию среднего чека в зависимости от района.
- Сформулировать итоговый вывод и рекомендации для заказчика.

## 0.2 Описание данных

Данные основаны на информации сервисов Яндекс Карты и Яндекс Бизнес за лето 2022 года. Информация носит исключительно справочный характер.

**Файл `rest_info.csv` — информация о заведениях:**

- `name` — название заведения;
- `address` — адрес заведения;
- `district` — административный район, в котором находится заведение;
- `category` — категория заведения (например, «кафе», «пиццерия», «кофейня»);
- `hours` — информация о днях и часах работы;
- `rating` — рейтинг заведения по оценкам пользователей в Яндекс Картах (высшая оценка — 5.0);
- `chain` — является ли заведение сетевым: `0` — нет, `1` — да (для маленьких сетей возможны ошибки);
- `seats` — количество посадочных мест.

**Файл `rest_price.csv` — информация о среднем чеке:**

- `price` — категория цен в заведении (например, «средние», «ниже среднего», «выше среднего»);
- `avg_bill` — средняя стоимость заказа в виде диапазона (например, «Средний счёт: 1000–1500 ₽», «Цена чашки капучино: 130–220 ₽»);
- `middle_avg_bill` — числовая оценка среднего чека (только для строк, начинающихся с «Средний счёт»);
- `middle_coffee_cup` — числовая оценка цены чашки капучино (только для строк, начинающихся с «Цена одной чашки капучино»).

## 0.3 Содержимое проекта

**Шаг 1. Загрузка данных и знакомство с ними**
- Загрузить датасеты `rest_info.csv` и `rest_price.csv`
- Изучить общую информацию о данных
- Проверить соответствие данных описанию, пропуски и типы данных
- Объединить датасеты в один

**Шаг 2. Предобработка данных**
- Привести типы данных к корректным
- Изучить и обработать пропущенные значения
- Проверить данные на явные и неявные дубликаты
- Создать столбец `is_24_7` (работа 24/7)

**Шаг 3. Исследовательский анализ данных**
- Категории заведений и их количество
- Распределение заведений по административным районам
- Сетевые и несетевые заведения
- Количество посадочных мест и выбросы
- Рейтинги заведений и их корреляции
- Топ-15 популярных сетей Москвы
- Средний чек в зависимости от района

**Шаг 4. Итоговый вывод и рекомендации**

# 1. Загрузка данных и знакомство с ними

Начнём с загрузки данных и знакомства с их структурой. На этом этапе важно понять объём данных, соответствие описанию, наличие пропусков и корректность типов — это определит дальнейшие шаги предобработки.

In [211]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [212]:
read_info = pd.read_csv('https://code.s3.yandex.net/datasets/rest_info.csv')
read_price = pd.read_csv('https://code.s3.yandex.net/datasets/rest_price.csv')

In [213]:
display(read_info.head())
display(read_info.tail())
print(read_info.info())


,id,name,category,address,district,hours,rating,chain,seats
0,0c3e3439a8c64ea5bf6ecd6ca6ae19f0,WoWфли,кафе,"Москва, улица Дыбенко, 7/1",Северный административный округ,"ежедневно, 10:00–22:00",5.0,0,NaN
1,045780ada3474c57a2112e505d74b633,Четыре комнаты,ресторан,"Москва, улица Дыбенко, 36, корп. 1",Северный административный округ,"ежедневно, 10:00–22:00",4.5,0,4.0
2,1070b6b59144425896c65889347fcff6,Хазри,кафе,"Москва, Клязьминская улица, 15",Северный административный округ,"пн-чт 11:00–02:00; пт,сб 11:00–05:00; вс 11:00...",4.6,0,45.0
3,03ac7cd772104f65b58b349dc59f03ee,Dormouse Coffee Shop,кофейня,"Москва, улица Маршала Федоренко, 12",Северный административный округ,"ежедневно, 09:00–22:00",5.0,0,NaN
4,a163aada139c4c7f87b0b1c0b466a50f,Иль Марко,пиццерия,"Москва, Правобережная улица, 1Б",Северный административный округ,"ежедневно, 10:00–22:00",5.0,1,148.0


,id,name,category,address,district,hours,rating,chain,seats
8401,0342ad1a45ed41ba89dcba246a8267e5,Суши Мания,кафе,"Москва, Профсоюзная улица, 56",Юго-Западный административный округ,"ежедневно, 09:00–02:00",4.4,0,86.0
8402,ee6bb7c3650e47bd8186fca08eda1091,Миславнес,кафе,"Москва, Пролетарский проспект, 19, корп. 1",Южный административный округ,"ежедневно, 08:00–22:00",4.8,0,150.0
8403,62e8c64d4c89467aba608e39ef87616b,Самовар,кафе,"Москва, Люблинская улица, 112А, стр. 1",Юго-Восточный административный округ,"ежедневно, круглосуточно",3.9,0,150.0
8404,06a0db5ecd4842d48cd6350aa923e297,Чайхана Sabr,кафе,"Москва, Люблинская улица, 112А, стр. 1",Юго-Восточный административный округ,"ежедневно, круглосуточно",4.2,1,150.0
8405,9d56f8f7a7f74df1a925bc0fd2de8aae,Kebab Time,кафе,"Москва, Россошанский проезд, 6",Южный административный округ,"ежедневно, круглосуточно",3.9,0,12.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8406 entries, 0 to 8405
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        8406 non-null   object 
 1   name      8406 non-null   object 
 2   category  8406 non-null   object 
 3   address   8406 non-null   object 
 4   district  8406 non-null   object 
 5   hours     7870 non-null   object 
 6   rating    8406 non-null   float64
 7   chain     8406 non-null   int64  
 8   seats     4795 non-null   float64
dtypes: float64(2), int64(1), object(6)
memory usage: 591.2+ KB
None


In [214]:
display(read_price.head())
print(read_price.info())

,id,price,avg_bill,middle_avg_bill,middle_coffee_cup
0,045780ada3474c57a2112e505d74b633,выше среднего,Средний счёт:1500–1600 ₽,1550.0,NaN
1,1070b6b59144425896c65889347fcff6,средние,Средний счёт:от 1000 ₽,1000.0,NaN
2,03ac7cd772104f65b58b349dc59f03ee,NaN,Цена чашки капучино:155–185 ₽,NaN,170.0
3,a163aada139c4c7f87b0b1c0b466a50f,средние,Средний счёт:400–600 ₽,500.0,NaN
4,8a343546b24e4a499ad96eb7d0797a8a,средние,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4058 entries, 0 to 4057
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 4058 non-null   object 
 1   price              3315 non-null   object 
 2   avg_bill           3816 non-null   object 
 3   middle_avg_bill    3149 non-null   float64
 4   middle_coffee_cup  535 non-null    float64
dtypes: float64(2), object(3)
memory usage: 158.6+ KB
None


## Первичное знакомство с данными

По результатам загрузки и изучения общей информации о датасетах можно сделать следующие наблюдения.

### Таблица с информацией о заведениях (`rest_info`)

- В таблице хранится **9 столбцов** и **8 406 строк**;
- Пропуски обнаружены в столбцах `hours` и `seats`;
- Требуется понизить размерность столбца `rating` и `chain`;
- Столбцы `seats` необходимо привести к типу `int`. При этом для `seats` стоит предварительно изучить данные: тип `float64` скорее всего избыточен для количества посадочных мест.

### Таблица с информацией о ценах (`rest_price`)

- В таблице хранится **5 столбцов** и **4 058 строк**;
- Пропуски обнаружены в столбцах `price`, `avg_bill`, `middle_avg_bill`, `middle_coffee_cup`;


In [215]:
df = pd.merge(read_info, read_price, on='id', how='left')

In [216]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8406 entries, 0 to 8405
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 8406 non-null   object 
 1   name               8406 non-null   object 
 2   category           8406 non-null   object 
 3   address            8406 non-null   object 
 4   district           8406 non-null   object 
 5   hours              7870 non-null   object 
 6   rating             8406 non-null   float64
 7   chain              8406 non-null   int64  
 8   seats              4795 non-null   float64
 9   price              3315 non-null   object 
 10  avg_bill           3816 non-null   object 
 11  middle_avg_bill    3149 non-null   float64
 12  middle_coffee_cup  535 non-null    float64
dtypes: float64(4), int64(1), object(8)
memory usage: 853.9+ KB
None


## Шаг 2. Предобработка данных 

## 2.1 Изучение корректности типолв данных

In [217]:
df['rating'] = pd.to_numeric(df['rating'], downcast = 'float')
df['seats'] = pd.to_numeric(df['seats'], downcast = 'float')
df['chain'] = pd.to_numeric(df['chain'], downcast = 'integer')


In [218]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8406 entries, 0 to 8405
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 8406 non-null   object 
 1   name               8406 non-null   object 
 2   category           8406 non-null   object 
 3   address            8406 non-null   object 
 4   district           8406 non-null   object 
 5   hours              7870 non-null   object 
 6   rating             8406 non-null   float32
 7   chain              8406 non-null   int8   
 8   seats              4795 non-null   float32
 9   price              3315 non-null   object 
 10  avg_bill           3816 non-null   object 
 11  middle_avg_bill    3149 non-null   float64
 12  middle_coffee_cup  535 non-null    float64
dtypes: float32(2), float64(2), int8(1), object(8)
memory usage: 730.7+ KB
None


## 2.2 Изучение пропущенных значений 

In [219]:
round(df.isna().mean()*100,2)

id                    0.00
name                  0.00
category              0.00
address               0.00
district              0.00
hours                 6.38
rating                0.00
chain                 0.00
seats                42.96
price                60.56
avg_bill             54.60
middle_avg_bill      62.54
middle_coffee_cup    93.64
dtype: float64

## Обработка пропущенных значений

По результатам объединения данных были обнаружены пропуски в нескольких столбцах. Для каждого столбца принято отдельное решение с учётом того, как он будет использоваться в дальнейшем исследовании.

### Столбец `hours`

Принято решение пропуски в столбце `hours`, заменить индикатором 'Не указанно'

### Столбец `seats`

Пропуски в столбце `seats` заполним **медианным значением по категориям**. Это обосновано тем, что в дальнейшем будут исследоваться только выбросы и аномальные значения, а медиана по категории устойчива к выбросам и лучше отражает типичное количество посадочных мест для каждого типа заведения.

### Столбцы `price`, `avg_bill`, `middle_avg_bill`, `middle_coffee_cup`

- Пропуски в столбце `price` можно **восстановить при помощи столбца `middle_avg_bill`** — если известен средний чек, можно определить ценовую категорию заведения.
- Столбец `avg_bill` **заполнить не получится**: пропуски возникли из-за технического сбоя либо потому, что информация не была указана на картах.
- Столбец `middle_avg_bill` зависит от `avg_bill`, поэтому пропуски в нём предлагается заполнить **значениями-индикаторами**.
- Столбец `middle_coffee_cup` отсутствует по той же причине, что и `avg_bill`, — заполнение пропусков аналогично, значениями-индикаторами.

In [220]:
display(df[['category','seats']].head())
df['seats'] = df['seats'].fillna(df.groupby('category')['seats'].transform('median'))
display(df[['category','seats']].head())

,category,seats
0,кафе,NaN
1,ресторан,4.0
2,кафе,45.0
3,кофейня,NaN
4,пиццерия,148.0


,category,seats
0,кафе,60.0
1,ресторан,4.0
2,кафе,45.0
3,кофейня,80.0
4,пиццерия,148.0


In [221]:
display(df.groupby('price')['middle_avg_bill'].agg(['min','max']))


,min,max
price,,
высокие,0.0,35000.0
выше среднего,375.0,4500.0
низкие,90.0,600.0
средние,165.0,2150.0


**Все диапазоны накладываются друг на друга, поэтому точно заменить значения нельзя, поэтому принято решение отказатся от замены на значения и просто замена значений на индикатор '-1'**

In [222]:
df['price'] = df['price'].fillna(-1)
df['middle_avg_bill'] = df['middle_avg_bill'].fillna(-1)
df['middle_coffee_cup'] = df['middle_coffee_cup'].fillna(-1)
df['hours'] = df['hours'].fillna('Не указанно')
df['avg_bill'] = df['avg_bill'].fillna('Не указанно')


In [223]:
display(df.head())

,id,name,category,address,district,hours,rating,chain,seats,price,avg_bill,middle_avg_bill,middle_coffee_cup
0,0c3e3439a8c64ea5bf6ecd6ca6ae19f0,WoWфли,кафе,"Москва, улица Дыбенко, 7/1",Северный административный округ,"ежедневно, 10:00–22:00",5.0,0,60.0,-1,Не указанно,-1.0,-1.0
1,045780ada3474c57a2112e505d74b633,Четыре комнаты,ресторан,"Москва, улица Дыбенко, 36, корп. 1",Северный административный округ,"ежедневно, 10:00–22:00",4.5,0,4.0,выше среднего,Средний счёт:1500–1600 ₽,1550.0,-1.0
2,1070b6b59144425896c65889347fcff6,Хазри,кафе,"Москва, Клязьминская улица, 15",Северный административный округ,"пн-чт 11:00–02:00; пт,сб 11:00–05:00; вс 11:00...",4.6,0,45.0,средние,Средний счёт:от 1000 ₽,1000.0,-1.0
3,03ac7cd772104f65b58b349dc59f03ee,Dormouse Coffee Shop,кофейня,"Москва, улица Маршала Федоренко, 12",Северный административный округ,"ежедневно, 09:00–22:00",5.0,0,80.0,-1,Цена чашки капучино:155–185 ₽,-1.0,170.0
4,a163aada139c4c7f87b0b1c0b466a50f,Иль Марко,пиццерия,"Москва, Правобережная улица, 1Б",Северный административный округ,"ежедневно, 10:00–22:00",5.0,1,148.0,средние,Средний счёт:400–600 ₽,500.0,-1.0


**После всех преобразования в данных не осталось пропусков**

## 2.3 Изучение явных и не явных дубликатов 

In [224]:
df.duplicated().sum()

0

In [225]:
def dupl(name):
    print(f'Кол-во дупликатов до и после обработки по столбцу: {name}')
    print(df[name].nunique())
    df[name] = df[name].str.lower()
    print(df[name].nunique())
    print('\n')

In [226]:
dupl('name')
dupl('category')
dupl('district')
dupl('address')

Кол-во дупликатов до и после обработки по столбцу: name
5614
5512


Кол-во дупликатов до и после обработки по столбцу: category
8
8


Кол-во дупликатов до и после обработки по столбцу: district
9
9


Кол-во дупликатов до и после обработки по столбцу: address
5753
5752




**Были убранные явные и неявные дупликаты**